**3. LightGBM**
---
Propiedad de René Adarme Amado
---
Universidad Antonio Nariño - Tesis de Maestría en Hidrogeología Ambiental
Abril de 2025

# **Librerías**

In [1]:
# Importar librerías
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import cross_validate, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (accuracy_score, recall_score, roc_auc_score,
                             f1_score, make_scorer, confusion_matrix)
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)  # Para filtrar advertencias


# **Preprocesado**

In [2]:
# 1. Cargar y limpiar la base
base = pd.read_excel("Base_2025_v3.xlsx")
base.drop(columns=['ID', 'Tipo_Estructuras', 'Alturas'], inplace=True)

# Codificar variables categóricas
variables_numericas = base.select_dtypes(include=['float64', 'int64']).columns.tolist()
if 'Agua' in variables_numericas:
    variables_numericas.remove('Agua')
variables_categoricas = base.select_dtypes(include=['object']).columns.tolist()

# Separar datos en variables predictoras y objetivo
X = base.drop(columns=['Agua'])
y = base['Agua'].astype(int)

# 2. Preprocesador
preprocesador = ColumnTransformer([
    ('numericas', StandardScaler(), variables_numericas),
    ('categoricas', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), variables_categoricas)
])

# 3. Definir las métricas de evaluación
def especificidad_score(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    if cm.shape == (2, 2):
        return cm[0, 0] / (cm[0, 0] + cm[0, 1]) if (cm[0, 0] + cm[0, 1]) > 0 else 0
    return 0

metricas = {
    'Exactitud': make_scorer(accuracy_score),
    'Sensibilidad': make_scorer(recall_score),
    'Especificidad': make_scorer(especificidad_score),
    'Precision': 'precision',
    'AUC-ROC': 'roc_auc',
    'Puntaje F1': 'f1'
}

#**MODELO 1**

# **Buscar k óptimo**

In [6]:
# 4. Definir modelo base
modelo_lgbm_1 = ImbPipeline([
    ('preprocesador', preprocesador),
    ('smote', SMOTE(random_state=42)),
    ('clasificador', lgb.LGBMClassifier(
        random_state=42,           # Aleatoridad (semilla)
        verbose=-1,                #informacion mostrada en el entrenamiento. -1: nada; 0:Básica; 1: detallada
        objective='binary',        #tipo de problema a resolver
        boosting_type='gbdt',      #tipo de algoritmo de boosting a utilizar
        num_leaves=30,             #número máximo de hojas en cada árbol del modelo [20-100]
        learning_rate=0.01,        # tamaño del paso en cada iteración del boosting [0.01 - 0.1]
        feature_fraction=0.8,      #Proporción de características a considerar en cada iteración del boosting[0.5 - 1.0]
        bagging_fraction=0.8,      #Proporción de datos a utilizar en cada iteración del boosting [0.5 - 1.0]
        bagging_freq=5,            #Frecuencia con la que se realiza el bagging[0.5 - 1.0]
        min_data_in_leaf=15        # Número mínimo de datos que debe haber en una hoja de un árbol [10-100]
        ))
])

# Búsqueda de k óptimo
valores_k = list(range(3, 13))
resultados_k = {}

for k in valores_k:
    resultados_cv = cross_validate(modelo_lgbm_1, X, y, cv=k, scoring=metricas, return_train_score=True) # Usando cross_validate
    resultados_k[k] = {
        metrica: {
            'media_validacion': np.mean(resultados_cv['test_' + metrica]),
            'desviacion': np.std(resultados_cv['test_' + metrica])
        } for metrica in ['Precision', 'Sensibilidad', 'Puntaje F1', 'AUC-ROC']
    }

# Seleccionar el mejor k
mejor_k = sorted(resultados_k.items(), key=lambda x: (-x[1]['Precision']['media_validacion'], x[1]['Precision']['desviacion'],
                                                      -x[1]['Sensibilidad']['media_validacion'], x[1]['Sensibilidad']['desviacion']))[0][0]

print(f"\n Valor óptimo de k: {mejor_k}")


 Valor óptimo de k: 5


Criterios para escoger k óptimo:
*   Primero: Mayor precisión promedio (orden descendente)
*   Segundo: Menor desviación de exactitud
*   Tercero: Mayor sensibilidad promedio
*   Cuarto: Menor desviación de sensibilidad





# **Modelar con k óptimo**

In [7]:
# 5. Entrenar Modelo 1 con k óptimo
resultados_lgbm_1 = cross_validate(modelo_lgbm_1, X, y, cv=mejor_k, scoring=metricas, return_train_score=True)

print("\nResumen de Modelo 1 con el k óptimo en ambos conjuntos de datos:")
for metrica in ['Precision', 'Sensibilidad', 'Puntaje F1', 'AUC-ROC']:
    media_train = np.mean(resultados_lgbm_1['train_' + metrica])
    std_train = np.std(resultados_lgbm_1['train_' + metrica])
    media_test = np.mean(resultados_lgbm_1['test_' + metrica])
    std_test = np.std(resultados_lgbm_1['test_' + metrica])
    print(f"{metrica:<15} | Entrenamiento: {media_train:.4f} ± {std_train:.4f} | Validación: {media_test:.4f} ± {std_test:.4f}")



Resumen de Modelo 1 con el k óptimo en ambos conjuntos de datos:
Precision       | Entrenamiento: 0.9054 ± 0.0135 | Validación: 0.7293 ± 0.0398
Sensibilidad    | Entrenamiento: 0.8833 ± 0.0269 | Validación: 0.6565 ± 0.2818
Puntaje F1      | Entrenamiento: 0.8938 ± 0.0107 | Validación: 0.6535 ± 0.1760
AUC-ROC         | Entrenamiento: 0.9292 ± 0.0036 | Validación: 0.5379 ± 0.1078


# **MODELO 2**

In [8]:
# Modelos 2 con Grid Search
rangos_estimadores_lgbm2 = {
    2: [60, 90, 100, 300]  # Número de árboles a considerar (usa los mejores obtenidos en BA)
}

modelos_grid_lgbm2 = {}     # Diccionario que guardará los mejores hiperparametros
resultados_grid_lgbm2 = {}  # Diccionario que guardará los resultados de la evaluación de los modelos
i = 2 # i = orden de la secuencia de modelos

param_grid_lgbm = {        #Definir la grilla de hiperparámetros
    'clasificador__n_estimators': rangos_estimadores_lgbm2[i],
    'clasificador__max_depth': [5, 7, 9],
    'clasificador__learning_rate': [0.002, 0.0021],
    'clasificador__num_leaves': [17, 18, 19, 20,21],
    'clasificador__max_depth': [2, 3, 4],
    'clasificador__min_data_in_leaf': [5, 6,7,8,9],
    'clasificador__feature_fraction': [0.8, 0.9],
    'clasificador__bagging_fraction': [0.6, 0.7, 0.8],
}

modelo_base_lgbm2 = ImbPipeline([
    ('preprocesador', preprocesador),
    ('smote', SMOTE(random_state=42)),
    ('clasificador', lgb.LGBMClassifier(
        random_state=42,           # Aleatoridad (semilla)
        verbose=-1,                #informacion mostrada en el entrenamiento. -1: nada; 0:Básica; 1: detallada
        objective='binary',        #tipo de problema a resolver
        boosting_type='gbdt',      #tipo de algoritmo de boosting a utilizar
        ))
])

grid_lgbm2 = GridSearchCV(modelo_base_lgbm2, param_grid=param_grid_lgbm, cv=mejor_k, scoring='precision', n_jobs=-1)  # Objeto GridSearchCV que buscará la mejor combinación de hiperparámetros
grid_lgbm2.fit(X, y)  # entrenar Grid Search

print(f"\n--- Modelo {i} ---")
print("Mejores hiperparámetros:", grid_lgbm2.best_params_)

resultados = cross_validate(grid_lgbm2.best_estimator_, X, y, cv=mejor_k, scoring=metricas, return_train_score=True)
resultados_grid_lgbm2[i] = resultados
modelos_grid_lgbm2[i] = grid_lgbm2.best_estimator_

print(f"\nResumen de métricas para Modelo {i} (k = {mejor_k}):\n")
print(f"{'Métrica':<15}{'Conjunto':<15}{'Media':<10}{'Desviación'}")
for metrica in ['Precision', 'Sensibilidad', 'Puntaje F1', 'AUC-ROC']:
    media_train = np.mean(resultados['train_' + metrica])
    std_train = np.std(resultados['train_' + metrica])
    media_val = np.mean(resultados['test_' + metrica])
    std_val = np.std(resultados['test_' + metrica])
    print(f"{metrica:<15}{'Entrenamiento':<15}{media_train:.4f}   {std_train:.4f}")
    print(f"{metrica:<15}{'Validación':<15}{media_val:.4f}   {std_val:.4f}")


--- Modelo 2 ---
Mejores hiperparámetros: {'clasificador__bagging_fraction': 0.6, 'clasificador__feature_fraction': 0.8, 'clasificador__learning_rate': 0.0021, 'clasificador__max_depth': 3, 'clasificador__min_data_in_leaf': 7, 'clasificador__n_estimators': 100, 'clasificador__num_leaves': 17}

Resumen de métricas para Modelo 2 (k = 5):

Métrica        Conjunto       Media     Desviación
Precision      Entrenamiento  0.8514   0.0372
Precision      Validación     0.8027   0.1062
Sensibilidad   Entrenamiento  0.7042   0.1025
Sensibilidad   Validación     0.5956   0.2891
Puntaje F1     Entrenamiento  0.7659   0.0565
Puntaje F1     Validación     0.6441   0.2412
AUC-ROC        Entrenamiento  0.7926   0.0447
AUC-ROC        Validación     0.5924   0.1800


# **MODELO 3**

Explora diferentes cantidades de árboles y reduce rango de hiperparámetros basado en resultados del modelo 2

In [11]:
# Modelos 3 con Grid Search
rangos_estimadores_lgbm3 = {
    3: [50, 120, 150, 200]  # Número de árboles a considerar
}

# Definir diccionarios
modelos_grid_lgbm3 = {}     # Guardar los mejores hiperparametros
resultados_grid_lgbm3 = {}  # Guardar los resultados de la evaluación de los modelos
i = 3 # i = orden en la secuencia de modelos

# Definir la grilla de hiperparámetros
param_grid_lgbm = {        #Definir la grilla de hiperparámetros
    'clasificador__n_estimators': rangos_estimadores_lgbm3[i],
    'clasificador__max_depth': [5, 7, 9],
    'clasificador__learning_rate': [0.001, 0.0015, 0.002],
    'clasificador__num_leaves': [16, 17, 18],
    'clasificador__max_depth': [2, 4, 6, 8],
    'clasificador__min_data_in_leaf': [5, 6, 7],
    'clasificador__feature_fraction': [0.8, 0.9],
    'clasificador__bagging_fraction': [0.5, 0.6],
}

# Crear el modelo base
modelo_base_lgbm3 = ImbPipeline([
    ('preprocesador', preprocesador),
    ('smote', SMOTE(random_state=42)),
    ('clasificador', lgb.LGBMClassifier(
        random_state=42,
        verbose=-1,
        objective='binary',        #tipo de problema a resolver
        boosting_type='gbdt',      #tipo de algoritmo de boosting a utilizar
        ))
])

# Buscar hiperparámetros Óptimos
grid_lgbm3 = GridSearchCV(modelo_base_lgbm3, param_grid=param_grid_lgbm, cv=mejor_k, scoring='precision', n_jobs=-1)
grid_lgbm3.fit(X, y)  # Entrenar el modelo

print(f"\n--- Modelo {i} ---")
print("Mejores hiperparámetros:", grid_lgbm2.best_params_)

resultados = cross_validate(grid_lgbm2.best_estimator_, X, y, cv=mejor_k, scoring=metricas, return_train_score=True)
resultados_grid_lgbm3[i] = resultados
modelos_grid_lgbm3[i] = grid_lgbm3.best_estimator_

print(f"\nResumen de métricas para Modelo {i} (k = {mejor_k}):\n")
print(f"{'Métrica':<15}{'Conjunto':<15}{'Media':<10}{'Desviación'}")
for metrica in ['Precision', 'Sensibilidad', 'Puntaje F1', 'AUC-ROC']:
    media_train = np.mean(resultados['train_' + metrica])
    std_train = np.std(resultados['train_' + metrica])
    media_val = np.mean(resultados['test_' + metrica])
    std_val = np.std(resultados['test_' + metrica])
    print(f"{metrica:<15}{'Entrenamiento':<15}{media_train:.4f}   {std_train:.4f}")
    print(f"{metrica:<15}{'Validación':<15}{media_val:.4f}   {std_val:.4f}")


--- Modelo 3 ---
Mejores hiperparámetros: {'clasificador__bagging_fraction': 0.6, 'clasificador__feature_fraction': 0.8, 'clasificador__learning_rate': 0.0021, 'clasificador__max_depth': 3, 'clasificador__min_data_in_leaf': 7, 'clasificador__n_estimators': 100, 'clasificador__num_leaves': 17}

Resumen de métricas para Modelo 3 (k = 5):

Métrica        Conjunto       Media     Desviación
Precision      Entrenamiento  0.8514   0.0372
Precision      Validación     0.8027   0.1062
Sensibilidad   Entrenamiento  0.7042   0.1025
Sensibilidad   Validación     0.5956   0.2891
Puntaje F1     Entrenamiento  0.7659   0.0565
Puntaje F1     Validación     0.6441   0.2412
AUC-ROC        Entrenamiento  0.7926   0.0447
AUC-ROC        Validación     0.5924   0.1800


#**Comparar modelos 1, 2 y 3**

Criterios de comparación: 1. La mas alta precision y baja desvicion estandar. 2. La mas alta sensibilidad y baja desvicion estandar.

In [15]:
# Obtener las métricas de cada modelo, incluyendo la desviación estándar
precision_modelo_1 = np.mean(resultados_lgbm_1['test_Precision'])
sensibilidad_modelo_1 = np.mean(resultados_lgbm_1['test_Sensibilidad'])
std_precision_modelo_1 = np.std(resultados_lgbm_1['test_Precision'])
std_sensibilidad_modelo_1 = np.std(resultados_lgbm_1['test_Sensibilidad'])

precision_modelo_2 = np.mean(resultados_grid_lgbm2[2]['test_Precision'])
sensibilidad_modelo_2 = np.mean(resultados_grid_lgbm2[2]['test_Sensibilidad'])
std_precision_modelo_2 = np.std(resultados_grid_lgbm2[2]['test_Precision'])
std_sensibilidad_modelo_2 = np.std(resultados_grid_lgbm2[2]['test_Sensibilidad'])

precision_modelo_3 = np.mean(resultados_grid_lgbm3[3]['test_Precision'])
sensibilidad_modelo_3 = np.mean(resultados_grid_lgbm3[3]['test_Sensibilidad'])
std_precision_modelo_3 = np.std(resultados_grid_lgbm3[3]['test_Precision'])
std_sensibilidad_modelo_3 = np.std(resultados_grid_lgbm3[3]['test_Sensibilidad'])

# Comparar modelos priorizando precisión y sensibilidad con baja desviación estándar
mejor_modelo = 1

# Priorizar precisión con baja desviación estándar
if (precision_modelo_2 > precision_modelo_1 and std_precision_modelo_2 < std_precision_modelo_1):
    mejor_modelo = 2
elif (precision_modelo_3 > precision_modelo_1 and std_precision_modelo_3 < std_precision_modelo_1) or \
     (precision_modelo_3 > precision_modelo_2 and std_precision_modelo_3 < std_precision_modelo_2):
    mejor_modelo = 3

# Si la precisión es similar, priorizar sensibilidad con baja desviación estándar
if mejor_modelo == 1 and (sensibilidad_modelo_2 > sensibilidad_modelo_1 and std_sensibilidad_modelo_2 < std_sensibilidad_modelo_1):
    mejor_modelo = 2
elif mejor_modelo == 1 and (sensibilidad_modelo_3 > sensibilidad_modelo_1 and std_sensibilidad_modelo_3 < std_sensibilidad_modelo_1) or \
     (sensibilidad_modelo_3 > sensibilidad_modelo_2 and std_sensibilidad_modelo_3 < std_sensibilidad_modelo_2) :
    mejor_modelo = 3

# Mostrar el resultado
print(f"De los modelos 1 al 3, el modelo de mejor rendimiento es el modelo: {mejor_modelo}")

De los modelos 1 al 3, el modelo de mejor rendimiento es el modelo: 1


# **MODELO 4**

Toma el mejor entre los modelos 1 a 3 y aplica regularización

In [16]:
# Definir la grilla de hiperparámetros (con regularizadores)
param_grid = {
    'clasificador__reg_alpha': [0.01, 0.1],
    'clasificador__reg_lambda': [0.01, 0.1]
}

# Definir el modelo base
modelo_lgbm_4 = ImbPipeline([
    ('preprocesador', preprocesador),
    ('smote', SMOTE(random_state=42)),
    ('clasificador', lgb.LGBMClassifier(
        random_state=42,
        verbose=-1,
        objective='binary',
        boosting_type='gbdt',
        num_leaves=30,
        learning_rate=0.01,
        feature_fraction=0.8,
        bagging_fraction=0.8,
        bagging_freq=5,
        min_data_in_leaf=15
        ))
])

# Crear el objeto GridSearchCV
grid_search = GridSearchCV(modelo_lgbm_4, param_grid, cv=mejor_k, scoring='precision', n_jobs=-1)

# Ajustar el modelo con la búsqueda de grilla
grid_search.fit(X, y)

# Obtener el mejor modelo con los regularizadores óptimos
mejor_modelo_lgbm_4 = grid_search.best_estimator_

# Mostrar los mejores hiperparámetros
print("\n--- Modelo 4 ---")
print("Mejores hiperparámetros:", grid_search.best_params_)

# Evaluar el mejor modelo
resultados_lgbm_4 = cross_validate(mejor_modelo_lgbm_4, X, y, cv=mejor_k, scoring=metricas, return_train_score=True)

# Imprimir el resumen de las métricas
print(f"\nResumen de métricas para Modelo 4 (k = {mejor_k}):\n")
print(f"{'Métrica':<15}{'Conjunto':<15}{'Media':<10}{'Desviación'}")
for metrica in ['Precision', 'Sensibilidad', 'Puntaje F1', 'AUC-ROC']:
    media_train = np.mean(resultados_lgbm_4['train_' + metrica])
    std_train = np.std(resultados_lgbm_4['train_' + metrica])
    media_val = np.mean(resultados_lgbm_4['test_' + metrica])
    std_val = np.std(resultados_lgbm_4['test_' + metrica])
    print(f"{metrica:<15}{'Entrenamiento':<15}{media_train:.4f}   {std_train:.4f}")
    print(f"{metrica:<15}{'Validación':<15}{media_val:.4f}   {std_val:.4f}")


--- Modelo 4 ---
Mejores hiperparámetros: {'clasificador__reg_alpha': 0.01, 'clasificador__reg_lambda': 0.01}

Resumen de métricas para Modelo 4 (k = 5):

Métrica        Conjunto       Media     Desviación
Precision      Entrenamiento  0.9046   0.0139
Precision      Validación     0.7326   0.0424
Sensibilidad   Entrenamiento  0.8813   0.0270
Sensibilidad   Validación     0.6592   0.2797
Puntaje F1     Entrenamiento  0.8924   0.0110
Puntaje F1     Validación     0.6572   0.1737
AUC-ROC        Entrenamiento  0.9288   0.0038
AUC-ROC        Validación     0.5361   0.1082


# **Resumen de modelos**

In [18]:
# Definir lista con los nombres de los modelos
nombres_modelos = ['Modelo 1', 'Modelo 2', 'Modelo 3', 'Modelo 4']

# Definir lista con las variables de resultados de cada modelo
resultados_modelos = [resultados_lgbm_1, resultados_grid_lgbm2[2], resultados_grid_lgbm3[3], resultados_lgbm_4]

# Definir las métricas a mostrar en el resumen
metricas = ['Precision', 'Sensibilidad', 'Puntaje F1', 'AUC-ROC']

# Mostrar encabezado
print(f"{'Modelo':<15}{'Métrica':<15}{'Entrenamiento':<15}{'Validación':<15}")

# Iterar sobre los modelos y las métricas
for i, modelo in enumerate(nombres_modelos):
    for metrica in metricas:
        media_train = np.mean(resultados_modelos[i]['train_' + metrica])
        std_train = np.std(resultados_modelos[i]['train_' + metrica])
        media_val = np.mean(resultados_modelos[i]['test_' + metrica])
        std_val = np.std(resultados_modelos[i]['test_' + metrica])
        print(f"{modelo:<15}{metrica:<15}{media_train:.4f} ± {std_train:.4f}   {media_val:.4f} ± {std_val:.4f}")

Modelo         Métrica        Entrenamiento  Validación     
Modelo 1       Precision      0.9054 ± 0.0135   0.7293 ± 0.0398
Modelo 1       Sensibilidad   0.8833 ± 0.0269   0.6565 ± 0.2818
Modelo 1       Puntaje F1     0.8938 ± 0.0107   0.6535 ± 0.1760
Modelo 1       AUC-ROC        0.9292 ± 0.0036   0.5379 ± 0.1078
Modelo 2       Precision      0.8514 ± 0.0372   0.8027 ± 0.1062
Modelo 2       Sensibilidad   0.7042 ± 0.1025   0.5956 ± 0.2891
Modelo 2       Puntaje F1     0.7659 ± 0.0565   0.6441 ± 0.2412
Modelo 2       AUC-ROC        0.7926 ± 0.0447   0.5924 ± 0.1800
Modelo 3       Precision      0.8514 ± 0.0372   0.8027 ± 0.1062
Modelo 3       Sensibilidad   0.7042 ± 0.1025   0.5956 ± 0.2891
Modelo 3       Puntaje F1     0.7659 ± 0.0565   0.6441 ± 0.2412
Modelo 3       AUC-ROC        0.7926 ± 0.0447   0.5924 ± 0.1800
Modelo 4       Precision      0.9046 ± 0.0139   0.7326 ± 0.0424
Modelo 4       Sensibilidad   0.8813 ± 0.0270   0.6592 ± 0.2797
Modelo 4       Puntaje F1     0.8924 ± 0.01